# Analytics for a better world
## Assignment 6: Linear programming (with indexed variables)

## Question 1 - Larry Edison (5 points)
Larry Edison is the director of the Computer Center for Buckly College. He now needs
to schedule the staffing of the center. It is open from 8 AM until midnight. Larry has
monitored the usage of the center at various times of the day and determined that
the following number of computer consultants are required:

Shift Time of Day Minimum Number of Consultants
A 8 AM - noon 4
B Noon - 4 PM 8
C 4 PM - 8 PM 10
D 8 PM - midnight 6

Two types of computer consultants can be hired: full-time and part-time.

Full-time consultants work 8 consecutive hours in one of the following shifts:
morning: 8 AM - 4 PM
afternoon: noon - 8 PM
evening: 4 PM - midnight
Full-time consultants are paid €40 per hour.

Part-time consultants can be hired to work any of the four 4-hour shifts listed in the
table above and are paid €30 per hour.

An additional requirement is that, during every time period, there must be at least two
full-time consultants on duty for every part-time consultant on duty.
Larry would like to determine how many full-time and part-time consultants should
work each shift in order to meet all staffing requirements at the minimum possible
cost.

Formulate an integer linear programming model with indexed variables and solve it
using Pyomo. Make sure that all decision variables can only take non-negative integer
values, since it is not possible to hire a fraction of a worker.

In [6]:
import pyomo.environ as pyo

In [7]:
### BEGIN SOLUTION
def model_larry_edison():
  model = pyo.ConcreteModel(name='larry_edison')

  model.full_time = pyo.Var(pyo.RangeSet(1, 3), domain=pyo.NonNegativeIntegers)
  model.part_time = pyo.Var(pyo.RangeSet(1, 4), domain=pyo.NonNegativeIntegers)

  full_time = model.full_time
  part_time = model.part_time

  @model.Objective(sense=pyo.minimize)
  def wage(model):
    return 320 * sum(full_time[i] for i in full_time) + 120 * sum(part_time[j] for j in part_time)

  @model.Constraint()
  def A_total(model):
    return full_time[1] + part_time[1] >= 4
  @model.Constraint()
  def B_total(model):
    return full_time[1] + full_time[2] + part_time[2] >= 8
  @model.Constraint()
  def C_total(model):
    return full_time[3] + full_time[2] + part_time[3] >= 10
  @model.Constraint()
  def D_total(model):
    return full_time[3] + part_time[4] >= 6

  @model.Constraint()
  def A_ratio(model):
    return full_time[1]>= 2 * part_time[1]
  @model.Constraint()
  def B_ratio(model):
    return (full_time[1] + full_time[2]) >= 2 * part_time[2]
  @model.Constraint()
  def C_ratio(model):
    return (full_time[3] + full_time[2]) >= 2 * part_time[3]
  @model.Constraint()
  def D_ratio(model):
    return full_time[3] >= 2 * part_time[4]
  return model
### END SOLUTION

In [15]:
if __name__ == "__main__":
    model = model_larry_edison()
    solver = pyo.SolverFactory("appsi_highs")
    results = solver.solve(model)
    pyo.assert_optimal_termination(results)

    print(f"Minimum daily wage: €{pyo.value(model.wage):,.0f}")

    full_time_names = {1: "Morning", 2: "Afternoon", 3: "Evening"}
    part_time_names = {1: "A", 2: "B", 3: "C", 4: "D"}

    print("\nFull-time consultants:")
    for i in model.full_time:
        print(f"  {full_time_names[i]}: {int(round(pyo.value(model.full_time[i])))}")

    print("\nPart-time consultants:")
    for j in model.part_time:
        print(f"  {part_time_names[j]}: {int(round(pyo.value(model.part_time[j])))}")

Minimum daily wage: €4,160

Full-time consultants:
  Morning: 3
  Afternoon: 3
  Evening: 4

Part-time consultants:
  A: 1
  B: 2
  C: 3
  D: 2


# Question 2 - Oxbridge (5 points)
Oxbridge University maintains a powerful mainframe computer for research use by its
faculty, Ph.D. students, and research associates. During all working hours, an operator
must be available to operate and maintain the computer, as well as to perform some
programming services. 

Beryl Ingram, the director of the computer facility, oversees
the operation. It is now the beginning of the fall semester, and Beryl is confronted
with the problem of assigning different working hours to her operators. Because all
the operators are currently enrolled in the university, they are available to work only a
limited number of hours each day. The table below shows, for each operator, both the
wage per hour and the maximum number of working hours that the operator is
available per week.

There are six operators (four undergraduate students and two graduate students).
They all have different wage rates because of differences in their experience with
computers and in their programming ability. Each operator is guaranteed a certain
minimum number of hours per week that will maintain an adequate knowledge of the
operation. This level is set at 8 hours per week for the undergraduate students (K.C.,
D.H., H.B., and S.C.) and 7 hours per week for the graduate students (K.S. and N.K.).
The computer facility is to be open for operation from 8 AM to 10 PM Monday through
Friday with exactly one operator on duty during each of these hours. Because of a
tight budget, Beryl has to minimize cost. She wishes to determine the number of
hours she should assign to each operator on each day. Formulate a linear
programming model for this problem.

In [22]:
import pyomo.environ as pyo
import pandas as pd
from io import StringIO

data = StringIO('''Operator;Wage;Mon;Tue;Wed;Thu;Fri;MinHours
K.C.; 25 ; 6 ; 0 ; 6 ; 0 ; 6 ; 8
D.H.; 26 ; 0 ; 6 ; 0 ; 6 ; 0 ; 8
H.B.; 24 ; 4 ; 8 ; 4 ; 0 ; 4 ; 8
S.C.; 23 ; 5 ; 5 ; 5 ; 0 ; 5 ; 8
K.S.; 28 ; 3 ; 0 ; 3 ; 8 ; 0 ; 7
N.K.; 30 ; 0 ; 0 ; 0 ; 6 ; 2 ; 7 ''')

df = pd.read_csv(data, sep=";", index_col='Operator')
days = ['Mon','Tue','Wed','Thu','Fri']

In [23]:
### BEGIN SOLUTION
def model_oxbridge():
    model = pyo.ConcreteModel(name="oxbridge")

    # First index: operator. Second index: day.
    model.x = pyo.Var(df.index, days, domain=pyo.NonNegativeIntegers)

    @model.Constraint(df.index)
    def min_hours(model, operator):
        return sum(model.x[operator, day] for day in days) >= df.loc[operator, "MinHours"]

    @model.Constraint(df.index, days)
    def availability(model, operator, day):
        return model.x[operator, day] <= df.loc[operator, day]

    @model.Constraint(days)
    def day_total(model, day):
        return sum(model.x[operator, day] for operator in df.index) == 14

    @model.Objective(sense=pyo.minimize)
    def wage(model):
        return sum(
            df.loc[operator, "Wage"] * model.x[operator, day]
            for operator in df.index
            for day in days
        )

    return model
### END SOLUTION

In [24]:
if __name__ == "__main__":
    model = model_oxbridge()
    solver = pyo.SolverFactory("appsi_highs")
    results = solver.solve(model)
    pyo.assert_optimal_termination(results)

    print(f"Minimum weekly wage: €{pyo.value(model.wage):,.0f}")
    print("Operator\t" + "\t".join(str(day) for day in days) + "\tTotal")
    for operator in df.index:
        hours = [int(round(pyo.value(model.x[operator, day]))) for day in days]
        print(str(operator) + "\t\t" + "\t".join(str(h) for h in hours) + "\t" + str(sum(hours)))

Minimum weekly wage: €1,755
Operator	Mon	Tue	Wed	Thu	Fri	Total
K.C.		4	0	2	0	3	9
D.H.		0	2	0	6	0	8
H.B.		4	7	4	0	4	19
S.C.		5	5	5	0	5	20
K.S.		1	0	3	3	0	7
N.K.		0	0	0	5	2	7
